# 工具的调用

直接调用

In [2]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息
    :param city: 城市名称
    :return: 天气信息字符串
    """
    return city + "：晴天，温度15℃"

In [3]:
get_weather.invoke({"city": "武汉"})

'武汉：晴天，温度15℃'

模型可以推理当前是否需要调用工具

In [18]:
# 工具的调用
from langchain_core.tools import tool

import os

from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

# 读取.env配置文件中的信息，相关环境变量以.env文件中优先
load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 模型初始化
llm_deepseek = ChatDeepSeek(
    model="deepseek-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL
)

@tool
def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息

    Args:
        city : 城市名称
    Returns:
        天气信息字符串
    """
    return city + "：晴天，温度15℃"

# 绑定工具
model_with_tools = llm_deepseek.bind_tools([get_weather])

# AI 可以决定是否调用工具
response = model_with_tools.invoke("武汉天气如何？")
# response = model_with_tools.invoke("2 + 3 = ?")

# 检查 AI 是否要调用工具
if response.tool_calls:
    print("AI 想调用工具：", response.tool_calls)
else:
    print("AI 直接回答：", response.content)

AI 想调用工具： [{'name': 'get_weather', 'args': {'city': '武汉'}, 'id': 'call_00_rEp5FN963tvmeHPXGM7A5597', 'type': 'tool_call'}]


In [12]:
# 工具的调用
from langchain_core.tools import tool

import os

from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

# 读取.env配置文件中的信息，相关环境变量以.env文件中优先
load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 模型初始化
llm_deepseek = ChatDeepSeek(
    model="deepseek-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL
)


# AI 可以决定是否调用工具
response = llm_deepseek.invoke("武汉天气如何？")
# response = model_with_tools.invoke("2 + 3 = ?")

# 检查 AI 是否要调用工具
if response.tool_calls:
    print("AI 想调用工具：", response.tool_calls)
else:
    print("AI 直接回答：", response.content)

AI 直接回答： 我无法实时查询武汉当前天气。一般来说，5月上旬武汉处于春末夏初，气温多在 **18～28℃** 左右，湿度较大，降雨较频繁，可能出现阵雨或雷阵雨，早晚偏凉、中午较热。

建议出门带伞，穿薄外套或长袖，并查看实时预报（如中国天气网、手机天气 App）。如果你告诉我具体日期，我可以帮你分析大致天气和穿衣建议。


In [9]:
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

from rich import print as rprint

import os

from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

# 读取.env配置文件中的信息，相关环境变量以.env文件中优先
load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 模型初始化
llm_deepseek = ChatDeepSeek(
    model="deepseek-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL
)

@tool
def get_weather(city: str) -> str:
    """
    获取指定城市的天气信息
    :param city: 城市名称
    :return: 天气信息字符串
    """
    return city + "：晴天，温度15℃"

# 绑定工具
model_with_tools = llm_deepseek.bind_tools([get_weather])

# 声明消息列表
messages = [
    HumanMessage("今天武汉天气如何")
]

# 模型生成调用工具请求
response = model_with_tools.invoke(messages)
rprint(response)

# 添加AIMessage到消息列表中
messages.append(response)

tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather":
        # 返回的是ToolMessage类型消息
        tool_response = get_weather.invoke(tool_call)
        print(type(tool_response))
        messages.append(tool_response)

print("==========> messages <==========")
for msg in messages:
    msg.pretty_print()

print("==========> messages <==========")
final_response = model_with_tools.invoke(messages)
print(f"final_response: \n{final_response}")

AIMessage(
    content='',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 'The user asks about the weather in Wuhan today. I should call get_weather with city 
"武汉".'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 60,
            'prompt_tokens': 308,
            'total_tokens': 368,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 21,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 128},
            'prompt_cache_hit_tokens': 128,
            'prompt_cache_miss_tokens': 180
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': '4a5d59fb-f06f-462f-9d34-558912281c10',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a0dc2d-21f6-7a52-a2a2-7f6870d4d982-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '武汉'},
            'id': 'call_00_c3NWuK7tgE0FPtdiyrFh0601',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 308,
        'output_tokens': 60,
        'total_tokens': 368,
        'input_token_details': {'cache_read': 128},
        'output_token_details': {'reasoning': 21}
    }
)

<class 'langchain_core.messages.tool.ToolMessage'>
==========> messages <==========
================================ Human Message =================================

今天武汉天气如何
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_00_c3NWuK7tgE0FPtdiyrFh0601)
 Call ID: call_00_c3NWuK7tgE0FPtdiyrFh0601
  Args:
    city: 武汉
================================= Tool Message =================================
Name: get_weather

武汉：晴天，温度15℃
==========> messages <==========
final_response: 
content='武汉今天天气：**晴天**，气温约 **15℃**。\n\n天气不错，适合外出活动。不过 15℃ 早晚可能偏凉，建议带一件外套注意保暖。如需了解未来几天的天气，或想查询其他城市，随时告诉我。' additional_kwargs={'refusal': None, 'reasoning_content': 'The tool returned the weather info. Let me report it.\n\nWait, I need to be careful - the tool output is a user message with the tool result. I should just report it.'} response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 387, 'total_tokens': 481, 'completion_tokens_de